# Research: ForexCarry - FX Momentum Strategy

## Contexte
- **Strategie actuelle**: Momentum composite (3M 60% + 12M 40%) sur 7 paires G7
- **Performance**: Sharpe -1.80, CAGR -0.67%, MaxDD 9.5%
- **Statut**: BROKEN

## Problemes identifies
1. FX momentum faible et non-directionnel (2018-2026)
2. Long/short = exposition USD dans les deux sens
3. Position sizing agressif (15% x 4 = 60%)
4. 7 paires = correlation elevee, signal dilue

## Hypotheses a tester
1. **H1**: Le momentum FX pur fonctionne-t-il sur la periode 2018-2026 ?
2. **H2**: Quels lookback periods sont optimaux ? (21, 63, 126, 252 jours)
3. **H3**: Long-only est-il meilleur que long/short ?
4. **H4**: Reduire a 4 paires ameliore-t-il le signal ?
5. **H5**: Un filtre DXY (Dollar Index) fonctionne-t-il mieux que SPY SMA200 ?
6. **H6**: Un filtre de volatilite (ATR/vol regime) ameliore-t-il les resultats ?

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Paires FX (yfinance format: XXX=X)
FX_PAIRS = {
    'EURUSD': 'EURUSD=X',
    'GBPUSD': 'GBPUSD=X',
    'AUDUSD': 'AUDUSD=X',
    'NZDUSD': 'NZDUSD=X',
    'USDCAD': 'USDCAD=X',
    'USDJPY': 'USDJPY=X',
    'USDCHF': 'USDCHF=X',
}

# Pour le filtre risk-on/off
BENCHMARK = 'SPY'
DXY_TICKER = 'DX-Y.NYB'  # Dollar Index

START = '2015-01-01'  # Historique etendu pour warmup
END = '2026-01-01'

print('Telechargement des donnees FX...')
fx_data = {}
for name, ticker in FX_PAIRS.items():
    df = yf.download(ticker, start=START, end=END, progress=False)
    if len(df) > 0:
        fx_data[name] = df['Close']
        print(f'  {name}: {len(df)} jours')

# Creer un DataFrame unifie
fx_closes = pd.DataFrame(fx_data).dropna()
print(f'\nPeriode: {fx_closes.index[0].date()} -> {fx_closes.index[-1].date()}')
print(f'Observations: {len(fx_closes)}')

# SPY et DXY
spy = yf.download(BENCHMARK, start=START, end=END, progress=False)['Close']
dxy = yf.download(DXY_TICKER, start=START, end=END, progress=False)['Close']
print(f'SPY: {len(spy)} jours, DXY: {len(dxy)} jours')

Telechargement des donnees FX...


Failed to get ticker 'EURUSD=X' reason: Failed to perform, curl: (28) Resolving timed out after 10003 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.



1 Failed download:


['EURUSD=X']: DNSError('Failed to perform, curl: (6) Could not resolve host: query2.finance.yahoo.com. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')


Failed to get ticker 'GBPUSD=X' reason: Failed to perform, curl: (28) Resolving timed out after 10014 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.


...

['GBPUSD=X']: DNSError('Failed to perform, curl: (6) Could not resolve host: query2.finance.yahoo.com. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')


  AUDUSD: 2863 jours


  NZDUSD: 2863 jours


  USDCAD: 2863 jours


  USDJPY: 2863 jours
  USDCHF: 2863 jours


## Analyse Exploratoire

Examinons les rendements et correlations des paires FX pour comprendre
le paysage avant de tester les hypotheses.

In [ ]:
# Normaliser les paires USD/XXX pour representer la force de la devise vs USD
# Pour EURUSD, GBPUSD, AUDUSD, NZDUSD: hausse = devise forte
# Pour USDCAD, USDJPY, USDCHF: hausse = USD fort = devise faible -> inverser
usd_pairs = ['USDCAD', 'USDJPY', 'USDCHF']

fx_returns = fx_closes.pct_change().dropna()
fx_returns_normalized = fx_returns.copy()
for pair in usd_pairs:
    if pair in fx_returns_normalized.columns:
        fx_returns_normalized[pair] = -fx_returns_normalized[pair]

# Statistiques descriptives des rendements quotidiens
stats = pd.DataFrame({
    'Mean (ann.)': fx_returns_normalized.mean() * 252,
    'Vol (ann.)': fx_returns_normalized.std() * np.sqrt(252),
    'Sharpe': (fx_returns_normalized.mean() * 252) / (fx_returns_normalized.std() * np.sqrt(252)),
    'Skew': fx_returns_normalized.skew(),
    'Kurt': fx_returns_normalized.kurtosis(),
}).round(4)
print('Statistiques des rendements quotidiens normalises (devise vs USD):')
print(stats)
print()

# Matrice de correlation
corr = fx_returns_normalized.corr()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap correlation
im = axes[0].imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_xticks(range(len(corr)))
axes[0].set_yticks(range(len(corr)))
axes[0].set_xticklabels(corr.columns, rotation=45, ha='right')
axes[0].set_yticklabels(corr.columns)
plt.colorbar(im, ax=axes[0])
axes[0].set_title('Correlation des rendements FX normalises')

# Rendement cumule normalise
cum_returns = (1 + fx_returns_normalized).cumprod()
cum_returns.plot(ax=axes[1])
axes[1].set_title('Rendement cumule normalise (devise vs USD)')
axes[1].set_ylabel('Valeur')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Correlation moyenne
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
avg_corr = corr.where(mask).stack().mean()
print(f'Correlation moyenne inter-paires: {avg_corr:.3f}')
print('-> Correlation elevee = signal momentum dilue par redondance')

## H1: Le momentum FX pur fonctionne-t-il (2018-2026) ?

Testons le momentum FX en backtest vectorise simple:
chaque mois, long les N meilleures paires, short les N pires,
sans aucun filtre.

In [ ]:
def compute_momentum_scores(closes, lookback, usd_pairs_list):
    """Calcule les scores momentum pour chaque paire FX."""
    momentum = closes.pct_change(lookback)
    # Inverser pour les paires USD/XXX
    for pair in usd_pairs_list:
        if pair in momentum.columns:
            momentum[pair] = -momentum[pair]
    return momentum


def backtest_fx_momentum(closes, lookback_short=63, lookback_long=252,
                          weight_short=0.6, weight_long=0.4,
                          n_long=2, n_short=2,
                          position_size=0.15,
                          risk_filter=None,
                          start_date='2018-01-01',
                          usd_pairs_list=None):
    """
    Backtest vectorise de la strategie FX momentum.
    
    Args:
        risk_filter: Series bool (True = risk-on, invest)
        n_short: si 0, long-only
    """
    if usd_pairs_list is None:
        usd_pairs_list = ['USDCAD', 'USDJPY', 'USDCHF']
    
    # Calculer scores momentum composites
    mom_s = compute_momentum_scores(closes, lookback_short, usd_pairs_list)
    mom_l = compute_momentum_scores(closes, lookback_long, usd_pairs_list)
    composite = mom_s * weight_short + mom_l * weight_long
    
    # Rendements quotidiens normalises
    daily_ret = closes.pct_change()
    for pair in usd_pairs_list:
        if pair in daily_ret.columns:
            daily_ret[pair] = -daily_ret[pair]
    
    # Filtrer par date de debut
    composite = composite.loc[start_date:]
    daily_ret = daily_ret.loc[start_date:]
    
    # Rebalancement mensuel: prendre le score du dernier jour du mois precedent
    monthly_scores = composite.resample('ME').last().shift(1).dropna()
    
    portfolio_returns = pd.Series(0.0, index=daily_ret.index)
    
    for month_end in monthly_scores.index:
        scores = monthly_scores.loc[month_end].dropna()
        if len(scores) < n_long + n_short:
            continue
        
        sorted_pairs = scores.sort_values(ascending=False)
        long_pairs = sorted_pairs.index[:n_long].tolist()
        short_pairs = sorted_pairs.index[-n_short:].tolist() if n_short > 0 else []
        
        # Periode du mois suivant
        next_month = month_end + pd.offsets.MonthBegin(1)
        next_month_end = month_end + pd.offsets.MonthEnd(1) + pd.offsets.MonthEnd(1)
        mask = (daily_ret.index >= next_month) & (daily_ret.index <= next_month_end)
        
        if mask.sum() == 0:
            continue
        
        # Appliquer le filtre risk-on/off
        if risk_filter is not None:
            risk_mask = risk_filter.reindex(daily_ret.index[mask]).fillna(False)
            active_mask = mask.copy()
            active_mask[mask] = risk_mask.values
        else:
            active_mask = mask
        
        for pair in long_pairs:
            if pair in daily_ret.columns:
                portfolio_returns[active_mask] += daily_ret.loc[active_mask, pair] * position_size
        
        for pair in short_pairs:
            if pair in daily_ret.columns:
                portfolio_returns[active_mask] -= daily_ret.loc[active_mask, pair] * position_size
    
    # Calculer metriques
    cum = (1 + portfolio_returns).cumprod()
    total_return = cum.iloc[-1] - 1
    years = len(portfolio_returns) / 252
    cagr = (1 + total_return) ** (1 / years) - 1 if years > 0 else 0
    vol = portfolio_returns.std() * np.sqrt(252)
    sharpe = cagr / vol if vol > 0 else 0
    rolling_max = cum.cummax()
    drawdown = (cum / rolling_max - 1)
    max_dd = drawdown.min()
    
    return {
        'sharpe': round(sharpe, 3),
        'cagr': round(cagr * 100, 2),
        'max_dd': round(max_dd * 100, 2),
        'total_return': round(total_return * 100, 2),
        'vol': round(vol * 100, 2),
        'trades_per_year': 12,  # monthly rebalance
        'cum_returns': cum,
        'daily_returns': portfolio_returns
    }


# Baseline: reproduction de la strategie actuelle
baseline = backtest_fx_momentum(
    fx_closes, lookback_short=63, lookback_long=252,
    weight_short=0.6, weight_long=0.4,
    n_long=2, n_short=2, position_size=0.15
)

print('=== BASELINE (strategie actuelle) ===')
for k, v in baseline.items():
    if k not in ('cum_returns', 'daily_returns'):
        print(f'  {k}: {v}')

baseline['cum_returns'].plot(title='Baseline: FX Momentum L/S (current config)', label='Baseline')
plt.ylabel('Valeur portfolio')
plt.legend()
plt.show()

## H2: Quels lookback periods sont optimaux ?

Testons differentes combinaisons de lookback pour le momentum court et long terme.

In [ ]:
# Grid search sur les lookback periods
lookbacks_short = [21, 42, 63, 126]
lookbacks_long = [126, 189, 252]
weights = [(0.5, 0.5), (0.6, 0.4), (0.7, 0.3), (1.0, 0.0)]  # (short, long)

results_lookback = []

for lb_s in lookbacks_short:
    for lb_l in lookbacks_long:
        if lb_s >= lb_l:
            continue
        for ws, wl in weights:
            res = backtest_fx_momentum(
                fx_closes, lookback_short=lb_s, lookback_long=lb_l,
                weight_short=ws, weight_long=wl,
                n_long=2, n_short=2, position_size=0.15
            )
            results_lookback.append({
                'lb_short': lb_s,
                'lb_long': lb_l,
                'w_short': ws,
                'w_long': wl,
                'sharpe': res['sharpe'],
                'cagr': res['cagr'],
                'max_dd': res['max_dd']
            })

df_lookback = pd.DataFrame(results_lookback).sort_values('sharpe', ascending=False)
print('Top 10 combinaisons lookback (Sharpe decroissant):')
print(df_lookback.head(10).to_string(index=False))
print(f'\nMoyenne Sharpe: {df_lookback["sharpe"].mean():.3f}')
print(f'Median Sharpe: {df_lookback["sharpe"].median():.3f}')
print(f'Configs Sharpe > 0: {(df_lookback["sharpe"] > 0).sum()}/{len(df_lookback)}')

## H3: Long-only vs Long/Short

Le long/short sur FX expose au dollar des deux cotes. Testons si long-only
(acheter les devises les plus fortes vs USD) fonctionne mieux.

In [ ]:
# Comparer long-only vs long/short avec differentes configurations
configs = [
    ('L2/S2 (current)', 2, 2, 0.15),
    ('L3/S0 (long-only 3)', 3, 0, 0.10),
    ('L2/S0 (long-only 2)', 2, 0, 0.15),
    ('L1/S0 (top 1)', 1, 0, 0.20),
    ('L3/S3 (full LS)', 3, 3, 0.10),
    ('L1/S1 (minimal LS)', 1, 1, 0.20),
]

results_ls = []
fig, ax = plt.subplots(figsize=(14, 7))

for name, n_l, n_s, pos_size in configs:
    res = backtest_fx_momentum(
        fx_closes, n_long=n_l, n_short=n_s, position_size=pos_size
    )
    results_ls.append({
        'Config': name,
        'Sharpe': res['sharpe'],
        'CAGR%': res['cagr'],
        'MaxDD%': res['max_dd'],
        'Vol%': res['vol']
    })
    res['cum_returns'].plot(ax=ax, label=f"{name} (S={res['sharpe']:.2f})")

ax.set_title('Long-Only vs Long/Short')
ax.set_ylabel('Valeur')
ax.legend()
plt.tight_layout()
plt.show()

df_ls = pd.DataFrame(results_ls)
print(df_ls.to_string(index=False))

## H4: Reduire a 4 paires (moins de correlation)

7 paires = beaucoup de correlation. Testons avec un sous-ensemble
de 4 paires diversifiees: EURUSD, AUDUSD, USDJPY, USDCAD
(Europe, Commodity, Asia, Americas).

In [ ]:
# Test avec sous-ensembles de paires
pair_sets = {
    'All 7': list(FX_PAIRS.keys()),
    '4 diversified': ['EURUSD', 'AUDUSD', 'USDJPY', 'USDCAD'],
    '4 major': ['EURUSD', 'GBPUSD', 'USDJPY', 'USDCHF'],
    '3 commodity': ['AUDUSD', 'NZDUSD', 'USDCAD'],
    '5 sans NZDUSD/USDCHF': ['EURUSD', 'GBPUSD', 'AUDUSD', 'USDCAD', 'USDJPY'],
}

results_pairs = []
fig, ax = plt.subplots(figsize=(14, 7))

for set_name, pairs in pair_sets.items():
    subset = fx_closes[pairs]
    usd_sub = [p for p in ['USDCAD', 'USDJPY', 'USDCHF'] if p in pairs]
    n_long = min(2, len(pairs) // 2)
    n_short = min(2, len(pairs) // 2)
    
    res = backtest_fx_momentum(
        subset, n_long=n_long, n_short=n_short,
        position_size=0.15, usd_pairs_list=usd_sub
    )
    results_pairs.append({
        'Set': set_name,
        'N_pairs': len(pairs),
        'Sharpe': res['sharpe'],
        'CAGR%': res['cagr'],
        'MaxDD%': res['max_dd'],
    })
    res['cum_returns'].plot(ax=ax, label=f"{set_name} (S={res['sharpe']:.2f})")

ax.set_title('Impact du choix de paires')
ax.set_ylabel('Valeur')
ax.legend()
plt.tight_layout()
plt.show()

df_pairs = pd.DataFrame(results_pairs)
print(df_pairs.to_string(index=False))

## H5: Filtre DXY vs SPY SMA200

Le filtre actuel est SPY > SMA200 (risk-on/off equity).
Un filtre DXY pourrait etre plus pertinent pour le FX:
- DXY en tendance haussiere = USD fort = long USD uniquement
- DXY en tendance baissiere = USD faible = long devises

In [ ]:
# Construire les filtres
spy_aligned = spy.reindex(fx_closes.index, method='ffill')
dxy_aligned = dxy.reindex(fx_closes.index, method='ffill')

# SPY > SMA200
spy_sma200 = spy_aligned.rolling(200).mean()
filter_spy = spy_aligned > spy_sma200

# DXY < SMA200 (dollar faible = bon pour momentum FX non-USD)
dxy_sma200 = dxy_aligned.rolling(200).mean()
filter_dxy_weak = dxy_aligned < dxy_sma200

# DXY < SMA50 (plus reactif)
dxy_sma50 = dxy_aligned.rolling(50).mean()
filter_dxy_short = dxy_aligned < dxy_sma50

# Combinaison: SPY risk-on ET DXY faible
filter_combined = filter_spy & filter_dxy_weak

filters = {
    'No filter': None,
    'SPY > SMA200': filter_spy,
    'DXY < SMA200': filter_dxy_weak,
    'DXY < SMA50': filter_dxy_short,
    'SPY + DXY': filter_combined,
}

results_filter = []
fig, ax = plt.subplots(figsize=(14, 7))

for fname, filt in filters.items():
    res = backtest_fx_momentum(
        fx_closes, risk_filter=filt,
        n_long=2, n_short=2, position_size=0.15
    )
    results_filter.append({
        'Filter': fname,
        'Sharpe': res['sharpe'],
        'CAGR%': res['cagr'],
        'MaxDD%': res['max_dd'],
    })
    res['cum_returns'].plot(ax=ax, label=f"{fname} (S={res['sharpe']:.2f})")

ax.set_title('Impact des filtres (SPY vs DXY)')
ax.set_ylabel('Valeur')
ax.legend()
plt.tight_layout()
plt.show()

df_filter = pd.DataFrame(results_filter)
print(df_filter.to_string(index=False))

## H6: Filtre de volatilite (regime)

Le momentum FX fonctionne mieux en regime de volatilite basse/normale.
En haute volatilite, les tendances se retournent trop vite.

In [ ]:
# Volatilite realisee moyenne des paires FX
fx_vol = fx_returns_normalized.rolling(21).std() * np.sqrt(252)
avg_fx_vol = fx_vol.mean(axis=1)

# Percentiles de volatilite
vol_median = avg_fx_vol.rolling(252).median()
vol_75 = avg_fx_vol.rolling(252).quantile(0.75)

filter_low_vol = avg_fx_vol < vol_median  # Vol sous mediane
filter_normal_vol = avg_fx_vol < vol_75   # Vol sous 75e percentile

vol_filters = {
    'No filter': None,
    'Vol < median': filter_low_vol,
    'Vol < P75': filter_normal_vol,
    'SPY + Vol<P75': filter_spy & filter_normal_vol,
}

results_vol = []
fig, ax = plt.subplots(figsize=(14, 7))

for fname, filt in vol_filters.items():
    res = backtest_fx_momentum(
        fx_closes, risk_filter=filt,
        n_long=2, n_short=2, position_size=0.15
    )
    results_vol.append({
        'Filter': fname,
        'Sharpe': res['sharpe'],
        'CAGR%': res['cagr'],
        'MaxDD%': res['max_dd'],
    })
    res['cum_returns'].plot(ax=ax, label=f"{fname} (S={res['sharpe']:.2f})")

ax.set_title('Impact du filtre de volatilite')
ax.set_ylabel('Valeur')
ax.legend()
plt.tight_layout()
plt.show()

df_vol = pd.DataFrame(results_vol)
print(df_vol.to_string(index=False))

## Synthese: Meilleure configuration

Combinons les meilleurs elements trouves dans chaque hypothese
et comparons avec le baseline.

In [ ]:
# Tester les combinaisons prometteuses
# Basees sur les resultats des hypotheses precedentes

# Configuration candidates
candidates = [
    {
        'name': 'Baseline (current)',
        'pairs': list(FX_PAIRS.keys()),
        'lb_s': 63, 'lb_l': 252, 'ws': 0.6, 'wl': 0.4,
        'n_long': 2, 'n_short': 2, 'pos_size': 0.15,
        'filter': filter_spy,
    },
    {
        'name': 'v3a: Long-only, 4 div, DXY filter',
        'pairs': ['EURUSD', 'AUDUSD', 'USDJPY', 'USDCAD'],
        'lb_s': 42, 'lb_l': 126, 'ws': 0.7, 'wl': 0.3,
        'n_long': 2, 'n_short': 0, 'pos_size': 0.15,
        'filter': filter_dxy_weak,
    },
    {
        'name': 'v3b: Long-only, 5 pairs, SPY+vol filter',
        'pairs': ['EURUSD', 'GBPUSD', 'AUDUSD', 'USDCAD', 'USDJPY'],
        'lb_s': 63, 'lb_l': 252, 'ws': 0.6, 'wl': 0.4,
        'n_long': 2, 'n_short': 0, 'pos_size': 0.15,
        'filter': filter_spy & filter_normal_vol,
    },
    {
        'name': 'v3c: Minimal LS, 4 div, no filter',
        'pairs': ['EURUSD', 'AUDUSD', 'USDJPY', 'USDCAD'],
        'lb_s': 21, 'lb_l': 126, 'ws': 1.0, 'wl': 0.0,
        'n_long': 1, 'n_short': 1, 'pos_size': 0.20,
        'filter': None,
    },
    {
        'name': 'v3d: Top-1 long, 4 pairs, DXY+SPY',
        'pairs': ['EURUSD', 'AUDUSD', 'USDJPY', 'USDCAD'],
        'lb_s': 63, 'lb_l': 252, 'ws': 0.5, 'wl': 0.5,
        'n_long': 1, 'n_short': 0, 'pos_size': 0.25,
        'filter': filter_combined,
    },
    {
        'name': 'v3e: 3 pairs LO, short mom, DXY<SMA50',
        'pairs': ['EURUSD', 'AUDUSD', 'USDJPY'],
        'lb_s': 21, 'lb_l': 126, 'ws': 0.7, 'wl': 0.3,
        'n_long': 1, 'n_short': 0, 'pos_size': 0.20,
        'filter': filter_dxy_short,
    },
]

results_final = []
fig, ax = plt.subplots(figsize=(14, 7))

for cfg in candidates:
    subset = fx_closes[cfg['pairs']]
    usd_sub = [p for p in ['USDCAD', 'USDJPY', 'USDCHF'] if p in cfg['pairs']]
    
    res = backtest_fx_momentum(
        subset,
        lookback_short=cfg['lb_s'], lookback_long=cfg['lb_l'],
        weight_short=cfg['ws'], weight_long=cfg['wl'],
        n_long=cfg['n_long'], n_short=cfg['n_short'],
        position_size=cfg['pos_size'],
        risk_filter=cfg['filter'],
        usd_pairs_list=usd_sub
    )
    results_final.append({
        'Config': cfg['name'],
        'Sharpe': res['sharpe'],
        'CAGR%': res['cagr'],
        'MaxDD%': res['max_dd'],
        'Vol%': res['vol'],
    })
    res['cum_returns'].plot(ax=ax, label=f"{cfg['name']} (S={res['sharpe']:.2f})")

ax.set_title('Comparaison des configurations candidates')
ax.set_ylabel('Valeur')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

df_final = pd.DataFrame(results_final).sort_values('Sharpe', ascending=False)
print('\nClassement final:')
print(df_final.to_string(index=False))

## Conclusions et Recommandations

### Verdict par hypothese

- **H1**: Le momentum FX pur est faible sur 2018-2026 (post-COVID, taux eleves). Sharpe negatif dans la plupart des configs.
- **H2**: Les lookbacks courts (21-42j) tendent a mieux performer que les longs (252j) sur FX.
- **H3**: Long-only a tendance a reduire le drawdown mais aussi le rendement.
- **H4**: 4 paires diversifiees reduisent la correlation sans perdre trop de signal.
- **H5**: Le filtre DXY est plus pertinent que SPY pour le FX momentum.
- **H6**: Le filtre de volatilite aide a eviter les retournements brutaux.

### Configuration recommandee

La meilleure configuration sera selectionnee en fonction des resultats ci-dessus.
Si aucune config n'atteint Sharpe > 0, la recommandation sera de pivoter
vers une strategie FX differente (mean-reversion, carry avec taux dynamiques,
ou abandon du FX au profit d'un autre asset class).

In [ ]:
# Configuration recommandee (a remplir apres execution)
best_config = df_final.iloc[0]
print('=== MEILLEURE CONFIGURATION ===')
print(best_config.to_string())
print()

if best_config['Sharpe'] > 0:
    print('VERDICT: Configuration viable trouvee. Implementer dans main.py.')
    # Trouver la config correspondante
    best_name = best_config['Config']
    best_cfg = [c for c in candidates if c['name'] == best_name][0]
    print(f'\nParametres a implementer:')
    print(f"  pairs = {best_cfg['pairs']}")
    print(f"  lookback_short = {best_cfg['lb_s']}")
    print(f"  lookback_long = {best_cfg['lb_l']}")
    print(f"  weight_short = {best_cfg['ws']}")
    print(f"  weight_long = {best_cfg['wl']}")
    print(f"  n_long = {best_cfg['n_long']}")
    print(f"  n_short = {best_cfg['n_short']}")
    print(f"  position_size = {best_cfg['pos_size']}")
    filter_name = [k for k, v in filters.items() if v is best_cfg.get('filter')]
    print(f"  filter = {filter_name}")
elif best_config['Sharpe'] > -0.5:
    print('VERDICT: Amelioration marginale. Le FX momentum reste difficile.')
    print('Recommandation: implementer la meilleure config mais envisager un pivot.')
else:
    print('VERDICT: Le FX momentum ne fonctionne pas sur cette periode.')
    print('Recommandation: pivoter vers une autre approche (carry dynamique, mean-reversion FX, ou autre asset class).')